# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL for the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata
metadata = dataset.metadata

# Show dataset overview
print("Dataset Title:", metadata.name)
print("Dataset Description:", metadata.description)
print("Dataset Identifier (@id):", metadata.id)
print("Published Date:", getattr(metadata, 'datePublished', ''))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets
print("Available record sets and their @id:")
for rs in record_sets:
    print("-", rs.id)

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs.id} (name='{getattr(rs, 'name', '')}')")
    print("Fields:")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name='{getattr(field, 'name', '')}', data_type='{getattr(field, 'data_type', '')}'")
    print("Columns:")
    for col in rs.columns:
        print(f"  Column @id: {col.id}, name='{getattr(col, 'name', '')}', field='{getattr(col, 'field', '')}'")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis using their `@id` references. Data is loaded dynamically using `mlcroissant`, and columns are referenced by their `@id`.

In [ ]:
# Prepare list of all record set @id
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load all record sets as DataFrames (if available)
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"DataFrame created for Record Set '@id': {rs_id}, columns: {df.columns.tolist()}")
        else:
            print(f"No records found for Record Set '@id': {rs_id}")
    except Exception as e:
        print(f"Error loading records for '@id': {rs_id}:", e)

# Preview main DataFrame (if available)
main_rs_id = record_sets_ids[0] if record_sets_ids else None
if main_rs_id and main_rs_id in dataframes:
    print("\nFirst five records:")
    display(dataframes[main_rs_id].head())
else:
    print("No DataFrame available for preview.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section provides examples using field and column `@id` references.

In [ ]:
# Example EDA: Select a numeric field (@id) from the dataset
# Find a numeric field for demonstration
selected_record_set_id = main_rs_id
if selected_record_set_id:
    fields = [field for field in dataset.record_set(selected_record_set_id).fields if field.data_type in ['Integer', 'Float', 'Number']]
    if fields:
        numeric_field_id = fields[0].id
        print(f"Using Numeric Field '@id': {numeric_field_id}")

        df = dataframes[selected_record_set_id]

        # Choose a threshold for filtering
        threshold = 10
        
        # Filter records (ensure field exists in DataFrame)
        if numeric_field_id in df.columns:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalize the numeric field
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / (filtered_df[numeric_field_id].std())
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try grouping by a categorical field
            group_fields = [f for f in dataset.record_set(selected_record_set_id).fields if f.data_type=='Text']
            if group_fields:
                group_field_id = group_fields[0].id
                if group_field_id in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field_id).mean()
                    print(f"Grouped data by {group_field_id}:")
                    display(grouped_df.head())
                else:
                    print(f"Group field '{group_field_id}' not found in DataFrame columns.")
            else:
                print("No categorical (Text) grouping field available.")
        else:
            print(f"Numeric field '@id': {numeric_field_id} not found in DataFrame columns.")
    else:
        print("No numeric fields found in main record set.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example using numeric and categorical field @id
if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Find numeric and categorical fields by @id
    numeric_fields = [f.id for f in dataset.record_set(selected_record_set_id).fields if f.data_type in ['Integer', 'Float', 'Number']]
    text_fields = [f.id for f in dataset.record_set(selected_record_set_id).fields if f.data_type == 'Text']
    if numeric_fields and text_fields:
        numeric_field_id = numeric_fields[0]
        group_field_id = text_fields[0]
        if numeric_field_id in df.columns:
            plt.figure(figsize=(8, 5))
            sns.histplot(df[numeric_field_id], bins=20, kde=True)
            plt.title(f"Distribution of {numeric_field_id}")
            plt.xlabel(numeric_field_id)
            plt.ylabel("Frequency")
            plt.show()

        if numeric_field_id in df.columns and group_field_id in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.xticks(rotation=45)
            plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.tight_layout()
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides rich clinicopathological and molecular details of second primary colorectal cancer in survivors.
- Exploration of record sets and fields by their `@id` enables precise reference and manipulation of the dataset.
- Filtering, normalization, grouping, and visualization steps were demonstrated using `mlcroissant` and standard Python tools.
- Further domain-specific analyses may be performed to investigate predictors and distribution of MSI-H phenotype.